# Kaggle Notebook Pipeline

**Run on [kaggle.com](https://www.kaggle.com) only** — not on your laptop.

1. **Settings → Internet → On** (needed for `git clone` + `pip install`)
2. **Settings → Accelerator → None** (CPU is enough for Phase 1)
3. **Add Data** → attach input dataset(s) listed in the config cell below
4. Run all cells, then **Save Version** → **Save as Dataset** to pass the compact
   `pipeline_output/data/` artifact to the next notebook

The repository is cloned into temporary storage and is never copied into the saved
notebook output. Pipeline stages write directly to the final output directory, avoiding
a second full-size copy at publish time.

**Inputs:** `RAW_EEG_INPUT` only.


In [ ]:
# --- Kaggle configuration (edit slugs to match your input datasets) ---
REPO_URL = "https://github.com/RandomPerson5571/ad_eeg.git"
REPO_BRANCH = "main"
PROJECT_DIR = "/kaggle/temp/ad_eeg"  # temporary; excluded from saved notebook output
OUTPUT_DIR = "/kaggle/working/pipeline_output"  # the only production artifact root

# Kaggle dataset slug with raw EEG (must contain EEG_data/dataset2/ and dataset3/)
RAW_EEG_INPUT = "REPLACE_WITH_RAW_EEG_DATASET_SLUG"

# Optional: output from a prior pipeline notebook (pipeline_output/data/, data/, or stage dirs at root)
PIPELINE_INPUT = None  # e.g. "REPLACE_WITH_PRIOR_PIPELINE_OUTPUT_SLUG"


In [ ]:
# --- Pipeline configuration (edit before running) ---
from pathlib import Path

MODE = "test"          # "inspect" | "test" | "full"
DATASET = "dataset2"   # "dataset2" | "dataset3" | "all"
EXPERIMENT = "baseline"
FORCE = False
WORKERS = 2
TEST_SUBJECTS = 5      # used when MODE == "test"
INSPECT_SUBJECT = 1    # subject number when MODE == "inspect"
KEEP_INTERMEDIATE_CHECKPOINTS = False  # False keeps epochs + QC/logs and saves substantial space

VALID_MODES = {"inspect", "test", "full"}
if MODE not in VALID_MODES:
    raise ValueError(f"MODE must be one of {VALID_MODES}, got {MODE!r}")

TEST_OUTPUT = Path("/kaggle/working/test_output")


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError(
        "This notebook runs on Kaggle only. "
        "Upload to kaggle.com, enable Internet, attach input datasets, then run."
    )

PROJECT_DIR = Path(PROJECT_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
# Notebook 01 defines MODE before setup. Its inspect/test checkpoints are scratch
# data; only their small reports under /kaggle/working/test_output are persisted.
if globals().get("MODE") in {"inspect", "test"}:
    OUTPUT_DIR = Path("/kaggle/temp/pipeline_output")
OUTPUT_DATA_DIR = OUTPUT_DIR / "data"
KAGGLE_WORKING_DIR = Path("/kaggle/working")


def _is_relative_to(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False


if _is_relative_to(PROJECT_DIR, KAGGLE_WORKING_DIR):
    raise ValueError(
        "PROJECT_DIR must be outside /kaggle/working so the Git clone is not "
        "included in the saved Kaggle output. Use /kaggle/temp/ad_eeg."
    )


def run(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


if not PROJECT_DIR.exists():
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {PROJECT_DIR}")

# Point the code's data/ path at the one-and-only persisted artifact tree.
# Preserve the small tracked seed files (for example data/manifest.json) first.
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
project_data = PROJECT_DIR / "data"
if project_data.is_symlink():
    if project_data.resolve() != OUTPUT_DATA_DIR.resolve():
        project_data.unlink()
elif project_data.exists():
    shutil.copytree(project_data, OUTPUT_DATA_DIR, dirs_exist_ok=True)
    shutil.rmtree(project_data)
if not project_data.exists():
    os.symlink(OUTPUT_DATA_DIR, project_data, target_is_directory=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
run(f"{sys.executable} -m pip install -q -r requirements-kaggle.txt", cwd=PROJECT_DIR)
print(f"Project root: {PROJECT_DIR.resolve()}", flush=True)
print(f"Pipeline output: {OUTPUT_DIR.resolve()}", flush=True)


def _tree_size(path: Path) -> int:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def _gib(n_bytes: int) -> float:
    return n_bytes / (1024 ** 3)


def _print_storage(label: str) -> None:
    usage = shutil.disk_usage(KAGGLE_WORKING_DIR)
    output_bytes = _tree_size(OUTPUT_DIR) if OUTPUT_DIR.exists() else 0
    print(
        f"{label}: output={_gib(output_bytes):.2f} GiB, "
        f"working free={_gib(usage.free):.2f} GiB",
        flush=True,
    )


def _pipeline_data_source(slug: str) -> Path:
    base = Path("/kaggle/input") / slug
    candidates = (base / "pipeline_output" / "data", base / "data", base)
    stage_dirs = {"audit", "preprocessed", "features", "models", "results"}
    for candidate in candidates:
        if candidate.is_dir() and any((candidate / name).exists() for name in stage_dirs):
            return candidate
    raise FileNotFoundError(
        f"Pipeline input '{slug}' has no pipeline_output/data/ or data/ artifact tree. "
        "Attach the previous notebook's saved output dataset."
    )


def _restore_pipeline_data(slug: str) -> None:
    pipeline_src = _pipeline_data_source(slug)
    source_bytes = _tree_size(pipeline_src)
    free_bytes = shutil.disk_usage(KAGGLE_WORKING_DIR).free
    reserve_bytes = 512 * 1024 ** 2
    if source_bytes + reserve_bytes > free_bytes:
        raise OSError(
            f"Pipeline input needs about {_gib(source_bytes):.2f} GiB but only "
            f"{_gib(free_bytes):.2f} GiB is free in /kaggle/working. "
            "Use a compact upstream artifact or start a fresh Kaggle session."
        )
    shutil.copytree(pipeline_src, OUTPUT_DATA_DIR, dirs_exist_ok=True)
    print(f"Restored pipeline data from {pipeline_src}", flush=True)
    _print_storage("After restore")


def summarize_output() -> None:
    if not OUTPUT_DATA_DIR.exists():
        print("No pipeline data was produced.", flush=True)
        return
    leaked_repos = [p for p in OUTPUT_DIR.rglob(".git") if p.is_dir()]
    if leaked_repos:
        raise RuntimeError(f"Refusing to publish a Git repository: {leaked_repos[0]}")
    n_files = sum(1 for p in OUTPUT_DATA_DIR.rglob("*") if p.is_file())
    _print_storage("Final artifact")
    print(f"Output ready: {OUTPUT_DIR} ({n_files} files)", flush=True)
    print(
        "Save Version → Save output as a new Kaggle Dataset, then attach it "
        "in the next notebook.",
        flush=True,
    )


def _find_eeg_root(slug: str) -> Path | None:
    base = Path("/kaggle/input") / slug
    if not base.exists():
        return None
    if (base / "EEG_data").is_dir():
        return base / "EEG_data"
    if (base / "dataset2").is_dir():
        return base
    for child in base.iterdir():
        if child.is_dir() and (child / "EEG_data").is_dir():
            return child / "EEG_data"
        if child.is_dir() and (child / "dataset2").is_dir():
            return child
    return None


eeg_link = PROJECT_DIR / "EEG_data"
if RAW_EEG_INPUT:
    src = _find_eeg_root(RAW_EEG_INPUT)
    if src is None:
        raise FileNotFoundError(
            f"Raw EEG not found for slug '{RAW_EEG_INPUT}'. "
            "Add Data → your dataset with EEG_data/dataset2/ and dataset3/."
        )
    if eeg_link.is_symlink():
        eeg_link.unlink()
    elif eeg_link.is_dir() and not eeg_link.is_symlink():
        pass
    elif eeg_link.exists():
        eeg_link.unlink()
    if not eeg_link.exists():
        os.symlink(src, eeg_link)
    print(f"EEG_data → {src}", flush=True)

if PIPELINE_INPUT:
    _restore_pipeline_data(PIPELINE_INPUT)


# 01 — Preprocessing

Mode-based preprocessing pipeline for fast iteration and production runs.

| Mode | Purpose | Output |
|------|---------|--------|
| **inspect** | One subject, interactive QC plots, debugging | `/kaggle/working/test_output/inspect/` |
| **test** | First N subjects, validation metrics, regression check | `/kaggle/working/test_output/test/` |
| **full** | Entire dataset, compact persisted artifacts | `pipeline_output/data/` → publish as Kaggle Dataset |

**Flow:** Configuration → Environment setup → Load config → Branch on `MODE` → (full only) validate the already-persisted artifacts. By default, completed subjects retain epochs, QC, and logs while large resumable intermediate FIF files are removed.


In [ ]:
from eeg.config import load_experiment, resolve_dataset
from eeg.repro import init_repro, snapshot_environment

dataset_specs = resolve_dataset(DATASET)
config = load_experiment(EXPERIMENT)

CONFIG = {
    "mode": MODE,
    "dataset": DATASET,
    "datasets": [ds.name for ds in dataset_specs],
    "experiment": EXPERIMENT,
    "force": FORCE,
    "workers": WORKERS,
    "test_subjects": TEST_SUBJECTS,
    "inspect_subject": INSPECT_SUBJECT,
    "seed": config.get("training", {}).get("random_state", 42),
}
repro = init_repro(CONFIG["seed"])
env = snapshot_environment()
print(CONFIG)


In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

from eeg.cli import subject_num_from_id
from eeg.config import experiment_metadata
from eeg.io import write_json
from eeg.paths import STAGE_SUFFIX, preprocessed_dir, qc_report_dir
from eeg.preprocess_report import write_preprocess_report
from eeg.qc import BAND_RANGES, backfill_spectral_qc, preprocessing_metrics
from eeg.runner import summarize_batch
from eeg.visualization import plot_preprocessing_panels_from_checkpoints
from scripts.preprocess_dataset import run_preprocess

QC_PLOTS = False  # set True for per-subject PNGs in full/test mode


def _subject_nums(ds):
    if MODE == "inspect":
        return [INSPECT_SUBJECT]
    return list(range(1, TEST_SUBJECTS + 1))


def _spectral_summary(metrics: dict) -> dict:
    """Extract µV²/Hz band-power fields for notebook summaries."""
    out = {}
    for band in BAND_RANGES:
        if metrics.get(f"{band}_before_uv2") is not None:
            out[f"{band}_before_uv2"] = metrics[f"{band}_before_uv2"]
        if metrics.get(f"{band}_delta_uv2") is not None:
            out[f"{band}_delta_uv2"] = metrics[f"{band}_delta_uv2"]
    return out


def _refresh_dataset_qc(ds, limit=None):
    """Backfill spectral QC from checkpoints and regenerate summary.csv."""
    patched = backfill_spectral_qc(ds.name, EXPERIMENT, limit=limit)
    report_paths = write_preprocess_report(
        ds.name, EXPERIMENT, config=config, qc_plots=QC_PLOTS, dataset_spec=ds
    )
    if patched:
        print(f"  [{ds.name}] backfilled spectral QC for {len(patched)} subject(s)")
    print(f"  [{ds.name}] QC report → {report_paths['summary_csv']}")
    return report_paths


def _compact_dataset_checkpoints(ds):
    """Remove resumable intermediates only after a valid epochs file exists."""
    root = preprocessed_dir(ds.name, EXPERIMENT)
    removed_files = 0
    removed_bytes = 0
    epoch_suffix = STAGE_SUFFIX["epochs"]
    for epoch_path in root.glob(f"*{epoch_suffix}"):
        participant_id = epoch_path.name[: -len(epoch_suffix)]
        for stage in ("raw", "filtered", "ica", "clean"):
            checkpoint = root / f"{participant_id}{STAGE_SUFFIX[stage]}"
            if checkpoint.is_file():
                removed_bytes += checkpoint.stat().st_size
                checkpoint.unlink()
                removed_files += 1
    print(
        f"  [{ds.name}] compacted {removed_files} intermediate checkpoints "
        f"({_gib(removed_bytes):.2f} GiB); epochs, QC, and logs retained"
    )


def _qc_subject(ds, subject_num, out_dir):
    metrics = preprocessing_metrics(ds, subject_num, EXPERIMENT)
    plot_path = plot_preprocessing_panels_from_checkpoints(
        ds, subject_num, EXPERIMENT, out_dir
    )
    alpha = metrics.get("alpha_before_uv2")
    alpha_d = metrics.get("alpha_delta_uv2")
    spectral = (
        f" alpha={alpha:.2f}µV² Δ={alpha_d:+.2f}"
        if alpha is not None and alpha_d is not None
        else ""
    )
    print(
        f"  {metrics['participant_id']}: bad_ch={metrics['n_bad_channels']} "
        f"rejected={metrics['n_epochs_rejected']}/{metrics['n_epochs_before_ar']}"
        f"{spectral} → {plot_path.name}"
    )
    return metrics, plot_path


summary = {
    "mode": MODE,
    "dataset": DATASET,
    "experiment": EXPERIMENT,
    "force": FORCE,
    "keep_intermediate_checkpoints": KEEP_INTERMEDIATE_CHECKPOINTS,
    "started_at": datetime.now(timezone.utc).isoformat(),
}
t0 = time.perf_counter()

if MODE == "inspect":
    out_root = TEST_OUTPUT / "inspect"
    out_root.mkdir(parents=True, exist_ok=True)
    summary["subjects"] = []

    for ds in dataset_specs:
        ds_out = out_root / ds.name
        ds_out.mkdir(parents=True, exist_ok=True)
        print(f"\n[{ds.name}] preprocess + inspect subject {INSPECT_SUBJECT}")
        run_preprocess(
            ds.name,
            EXPERIMENT,
            workers=1,
            force=FORCE,
            limit=None,
            subject=f"sub-{INSPECT_SUBJECT:03d}",
            qc_plots=True,
        )
        _refresh_dataset_qc(ds, limit=INSPECT_SUBJECT)
        metrics, _ = _qc_subject(ds, INSPECT_SUBJECT, ds_out)
        summary["subjects"].append({**metrics, **_spectral_summary(metrics)})
        summary["qc_report"] = str(qc_report_dir(ds.name, EXPERIMENT) / "summary.csv")

elif MODE == "test":
    out_root = TEST_OUTPUT / "test"
    out_root.mkdir(parents=True, exist_ok=True)
    summary["datasets"] = {}

    for ds in dataset_specs:
        ds_out = out_root / ds.name
        ds_out.mkdir(parents=True, exist_ok=True)
        print(f"\n[{ds.name}] preprocessing first {TEST_SUBJECTS} subjects...")
        results = run_preprocess(
            ds.name, EXPERIMENT, workers=WORKERS, force=FORCE, limit=TEST_SUBJECTS, qc_plots=QC_PLOTS
        )
        report_paths = _refresh_dataset_qc(ds, limit=TEST_SUBJECTS)
        batch = summarize_batch(results)
        summary["datasets"][ds.name] = {
            "completed": batch.completed,
            "skipped": batch.skipped,
            "failed": batch.failed,
            "qc_report": str(report_paths["summary_csv"]),
            "subjects": [
                {
                    "participant_id": r.log.get("participant_id"),
                    "status": r.status,
                    "runtime_seconds": r.log.get("runtime_seconds"),
                    "n_bad_channels": len(r.log.get("bad_channels", [])),
                    "n_epochs_rejected": r.log.get("n_epochs_rejected"),
                    **_spectral_summary(
                        preprocessing_metrics(
                            ds,
                            subject_num_from_id(r.log.get("participant_id")),
                            EXPERIMENT,
                        )
                    ),
                }
                for r in results
            ],
        }
        print(
            f"[{ds.name}] completed={batch.completed} "
            f"skipped={batch.skipped} failed={batch.failed}"
        )

        print(f"[{ds.name}] QC plots...")
        for sn in _subject_nums(ds):
            _qc_subject(ds, sn, ds_out)

elif MODE == "full":
    summary["datasets"] = {}
    all_results = []

    for ds in dataset_specs:
        print(f"\n[{ds.name}] full preprocessing run...")
        results = run_preprocess(
            ds.name, EXPERIMENT, workers=WORKERS, force=FORCE, limit=None, qc_plots=QC_PLOTS
        )
        report_paths = _refresh_dataset_qc(ds)
        batch = summarize_batch(results)
        all_results.extend(results)
        summary["datasets"][ds.name] = {
            "completed": batch.completed,
            "skipped": batch.skipped,
            "failed": batch.failed,
            "n_subjects": len(results),
            "qc_report": str(report_paths["summary_csv"]),
        }
        print(
            f"[{ds.name}] completed={batch.completed} "
            f"skipped={batch.skipped} failed={batch.failed}"
        )
        if not KEEP_INTERMEDIATE_CHECKPOINTS:
            _compact_dataset_checkpoints(ds)
        _print_storage(f"After {ds.name}")

    runtimes = [r.log.get("runtime_seconds", 0) for r in all_results if r.log.get("runtime_seconds")]
    summary["mean_runtime_seconds"] = round(sum(runtimes) / len(runtimes), 2) if runtimes else None
    summary["config"] = experiment_metadata(
        DATASET, EXPERIMENT, config, n_processed=len(all_results)
    )

else:
    raise ValueError(f"Unknown MODE: {MODE}")

summary["elapsed_seconds"] = round(time.perf_counter() - t0, 2)
summary["finished_at"] = datetime.now(timezone.utc).isoformat()

if MODE == "full":
    meta_path = Path("data") / "preprocess_full_summary.json"
else:
    meta_path = TEST_OUTPUT / f"preprocess_{MODE}_summary.json"
meta_path.parent.mkdir(parents=True, exist_ok=True)
write_json(meta_path, summary)
print(f"\nSummary → {meta_path}")
print(json.dumps(summary, indent=2, default=str))


In [ ]:
# Full-mode artifacts were written directly to the final output directory.
if MODE == "full" and Path("/kaggle/input").exists():
    summarize_output()
elif MODE != "full":
    print(f"MODE={MODE!r}: skipping Kaggle dataset publish (use MODE='full' for production output).")
